<a href="https://colab.research.google.com/github/karquiza/bikeshare-toronto-demand/blob/main/notebooks/05_integrating_weather_info.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Important note- I have not read the text in detail yet, will do it Monday - Aditi
#4. Operational Window Filtering, Hourly Aggregation, and Feature Integration
##4.1. Seasonal Window Selection (May 1st – October 31st)
Bike-sharing demand in Toronto exhibits extreme seasonality due to winter weather conditions, sub-zero temperatures, and snow accumulation. To build robust demand forecasting models, the dataset is filtered to restrict the analysis to peak operational months between May 1st and October 31st (month 05 to month 10).

Justification:

Behavioral Consistency: Winter months (November to April) introduce distinct extreme weather anomalies, reduced overall commuter volume, and altered trip intent (e.g., severe drop in casual ridership). Training models across a full annual cycle introduces severe variance and multimodal distributions that degrade forecasting performance during peak season.

Computational Efficiency: Reducing the dataset scope from full 12-month years to the 6 peak operational months filters out lower-value noise while retaining over 80% of total annual trip volume across the multi-year history, ensuring optimal performance for production model training.

##4.2. Time-Series Restructuring and Station-Level Aggregation
Raw bike-share trip logs represent individual transactions containing independent Start_Time, End_Time, Start_Station_Id, and End_Station_Id values. Machine learning algorithms for inventory rebalancing and capacity planning require structured time-series data aggregated at fixed interval time-steps per station.

Hourly Binning (timestamp): Timestamps for trip departures (Start_Time) and trip arrivals (End_Time) are floored to the nearest hourly boundary (dt.floor('h')).

Dual-Demand Tracking:

start_demand (Departures): Aggregates total trip originations grouped by start_hour and Start_Station_Id. This captures station outflow and potential stock depletion.

end_demand (Arrivals): Aggregates total trip completions grouped by end_hour and End_Station_Id. This captures station inflow and potential dock overcrowding.

Unified Outer Joining: An outer merge is executed on ['timestamp', 'station_id'] across the aggregated start and end counts. Missing entries are filled with 0.

Justification: Merging departures and arrivals under a single unified timestamp per station_id provides a complete net inventory change equation (
ΔN=Arrivals−Departures) for every station hour. Retaining missing station hours with zero counts prevents sampling bias towards high-volume hours, ensuring models accurately learn low-demand and off-peak states.

##4.3. Exogenous Weather Feature Integration
Weather parameters are primary dynamic drivers of micromobility demand. The hourly aggregated demand dataset is joined with localized hourly weather logs derived from Environment Canada monitoring stations.

Feature Alignment: The weather dataset (bike_stations_hourly_weather_analyzed.xlsx) is joined on timestamp and station_id using a left join.

Included Environmental Predictors: Key weather variables merged into the modeling table include temperature (
∘
 C), precipitation amounts (mm), wind speed (km/h), relative humidity (%), and weather descriptions (e.g., rain, clear, fog).

Justification: Temporal features alone (e.g., hour of day, day of week) cannot capture real-time demand shifts caused by adverse weather conditions. Integrating hourly localized weather metrics enables predictive models to adjust demand forecasts dynamically during rain events or temperature extremes.

In [4]:
# ==============================================================================
# EXTENSION of M3: May-October Filtering, Hourly Station Aggregation & Weather Integration
# ==============================================================================

import os
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/DataSets"    # shared Drive folder

# Path definition based on your directory structure setup
PROCESSED_OLD = f"{DATA}/processed-old"

def process_hourly_demand_may_to_oct(years, processed_folder):
    """
    Iterates through processed monthly files (May to October), loads demand data,
    and calculates hourly start and end demand per station.
    """
    all_station_hours = []

    for year in years:
        # Filter: Iterate only from May (5) through October (10)
        for month in range(5, 11):
            file_name = f"bikeshare_{year}_{month:02d}_demand.csv"
            file_path = os.path.join(processed_folder, file_name)

            if not os.path.exists(file_path):
                print(f"Warning: File not found - {file_path}")
                continue

            df = pd.read_csv(file_path, parse_dates=['Start_Time', 'End_Time'])

            # Ensure valid timestamps
            df = df[df['Start_Time'].notna()].copy()

            # Floor timestamps to hourly bins
            df['start_hour'] = df['Start_Time'].dt.floor('h')
            df['end_hour'] = df['End_Time'].dt.floor('h')

            # Calculate hourly start demand per station
            start_demand = (
                df.groupby(['start_hour', 'Start_Station_Id'])
                  .size()
                  .reset_index(name='start_demand')
                  .rename(columns={'start_hour': 'timestamp', 'Start_Station_Id': 'station_id'})
            )

            # Calculate hourly end demand per station
            end_df = df[df['End_Station_Id'].notna()].copy()
            end_demand = (
                end_df.groupby(['end_hour', 'End_Station_Id'])
                      .size()
                      .reset_index(name='end_demand')
                      .rename(columns={'end_hour': 'timestamp', 'End_Station_Id': 'station_id'})
            )

            # Outer merge start and end demand per station hour
            monthly_agg = pd.merge(
                start_demand,
                end_demand,
                on=['timestamp', 'station_id'],
                how='outer'
            ).fillna(0)

            monthly_agg['year'] = year
            monthly_agg['month'] = month

            all_station_hours.append(monthly_agg)
            print(f"Successfully processed: {file_name}")

    if not all_station_hours:
        raise FileNotFoundError("No files were successfully loaded. Please verify the directory path.")

    # Concatenate all monthly datasets into master DataFrame
    master_hourly = pd.concat(all_station_hours, ignore_index=True)
    return master_hourly

# Set years present in your dataset (e.g., [2024, 2025])
YEARS_TO_PROCESS = [2023,2024,2025]

master_hourly_demand = process_hourly_demand_may_to_oct(YEARS_TO_PROCESS, PROCESSED_OLD)
print("\nMaster Hourly Demand Shape (May-Oct):", master_hourly_demand.shape)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully processed: bikeshare_2023_05_demand.csv
Successfully processed: bikeshare_2023_06_demand.csv
Successfully processed: bikeshare_2023_07_demand.csv
Successfully processed: bikeshare_2023_08_demand.csv
Successfully processed: bikeshare_2023_09_demand.csv
Successfully processed: bikeshare_2023_10_demand.csv
Successfully processed: bikeshare_2024_05_demand.csv
Successfully processed: bikeshare_2024_06_demand.csv
Successfully processed: bikeshare_2024_07_demand.csv
Successfully processed: bikeshare_2024_08_demand.csv
Successfully processed: bikeshare_2024_09_demand.csv
Successfully processed: bikeshare_2024_10_demand.csv
Successfully processed: bikeshare_2025_05_demand.csv
Successfully processed: bikeshare_2025_06_demand.csv
Successfully processed: bikeshare_2025_07_demand.csv
Successfully processed: bikeshare_2025_08_demand.csv
Successfully processed:

In [5]:
# ------------------------------------------------------------------------------
# Weather Data Integration from Excel
# ------------------------------------------------------------------------------

WEATHER_FILE = f"{DATA}/bike_stations_hourly_weather_analyzed.xlsx"

if os.path.exists(WEATHER_FILE):
    # Read weather dataset directly from the DataSets root folder
    weather_df = pd.read_excel(WEATHER_FILE)

    # Print columns to verify contents
    print("Columns found in weather Excel file:", weather_df.columns.tolist())

    # Rename 'datetime' to 'timestamp' to match the demand dataset keys
    if 'datetime' in weather_df.columns:
        weather_df = weather_df.rename(columns={'datetime': 'timestamp'})

    # Ensure timestamp column is parsed as datetime
    if 'timestamp' in weather_df.columns:
        weather_df['timestamp'] = pd.to_datetime(weather_df['timestamp'])
    else:
        raise KeyError(f"Could not find a 'datetime' or 'timestamp' column in your Excel file. Columns are: {weather_df.columns.tolist()}")

    # Merge hourly demand with weather dataset on 'timestamp' and 'station_id'
    master_integrated = pd.merge(
        master_hourly_demand,
        weather_df,
        on=['timestamp', 'station_id'],
        how='left'
    )

    print("Successfully integrated weather data into the master dataset.")
    print("Final Integrated Dataset Shape:", master_integrated.shape)

    # Save the merged dataset back into profile directory
    OUTPUT_FILE = f"{DATA}/05-CodeOutput_station_hourly_demand_weather_may_oct.csv"
    master_integrated.to_csv(OUTPUT_FILE, index=False)
    print(f"Final modeling-ready dataset saved to: {OUTPUT_FILE}")
else:
    print(f"Error: Weather file not found at '{WEATHER_FILE}'. Please verify the file name and location.")

Columns found in weather Excel file: ['station_id', 'short_name', 'name', 'bike_lat', 'bike_lon', 'wx_station_id', 'wx_name_x', 'wx_lat', 'wx_lon', 'datetime', 'temp_c', 'dew_point_temp_c', 'relative_humidity_pct', 'precip_mm', 'wind_spd_kmh', 'wx_name_y']
Successfully integrated weather data into the master dataset.
Final Integrated Dataset Shape: (5681096, 20)
Final modeling-ready dataset saved to: /content/drive/MyDrive/DataSets/05-CodeOutput_station_hourly_demand_weather_may_oct.csv


##Integration and Final Dataset Architecture
###Overview of Our Weather and Demand Integration Pipeline
Our group successfully executed the integration of our hourly station-level bike demand data with the localized weather dataset, producing a modeling-ready master file. By aligning the temporal and spatial keys across our processed bike-share records and the weather analysis file (bike_stations_hourly_weather_analyzed.xlsx), we resolved structural differences—specifically mapping the weather file's datetime column to a unified timestamp key—to achieve a seamless merge.
###Key Components of Our Final Integrated Dataset
- Spatial and Temporal Granularity: Our merged dataset is structured at the individual station and hourly level. Each record maps a specific bike station (station_id) during a specific hour (timestamp), ensuring that our predictive models capture granular fluctuations in urban mobility rather than coarse daily averages.
- Comprehensive Feature Set (20 Columns): The final integrated dataset comprises 20 distinct features. These include spatial identifiers and station metadata (station_id, short_name, name, geographic coordinates), temporal markers, dual-direction demand metrics (start_demand and end_demand), and localized meteorological attributes.   
- Localized Meteorological Indicators: Rather than relying on city-wide generalized weather metrics, our dataset incorporates localized weather station assignments (wx_station_id, wx_lat, wx_lon) and specific hourly environmental metrics including temperature (temp_c), dew point temperature (dew_point_temp_c), relative humidity (relative_humidity_pct), precipitation (precip_mm), and wind speed (wind_spd_kmh).   
###Justification of Dataset Scale and Final Output Shape
- Dimensions Analysis (5,681,096 Rows $\times$ 20 Columns): The resulting scale of over 5.6 million rows reflects our comprehensive multi-station scope evaluated across the full peak operational window (May 1st to October 31st). This massive volume provides the necessary statistical depth for training robust machine learning and time-series forecasting models.   
- Preservation of Zero-Demand States: By utilizing an outer join approach during our hourly aggregation, our pipeline retains every active station-hour combination—even during periods of zero departures or zero arrivals. This prevents survivorship bias, ensuring our models accurately learn low-demand and off-peak operational states.
- Downstream Modeling Readiness: The final integrated file has been systematically exported and stored in our project directory as 05-CodeOutput_station_hourly_demand_weather_may_oct.csv. This artifact serves as the definitive foundation for our upcoming predictive modeling phase, allowing us to evaluate how exogenous variables like temperature and precipitation drive micro-mobility demand across Toronto's bike-share network.